# Phase 3 — Explanation Faithfulness Evaluation V1
**40-claim blinded evidence review | Reproducible sampling | No LLM/API/GPU**

This notebook uses the **original full-139 QC workbook** (sheets `Claim_Audit` and `Cited_Evidence`) containing 155 claims and cited evidence. It never exposes previous AI labels in the review file, never modifies the original reports, and **does not invent review labels**.

**Metric:** Fully supported claim rate among reviewed claims, with partial, unsupported and uncertain reported separately. This evaluates *support by retrieved evidence*, not factual truth against original Jira documents. A random stratified sample of 40 claims is **not a new independent test set**; disclose any earlier AI exposure.


## 1. Data access and paths
Required input: `Agentic_GraphRAG_Full139_Final_QC_Workbook.xlsx`; expected directory: `MyDrive/03_Investigation_Explanation/Explanation_Evaluation/Input_Data/`. You can alternatively set `INPUT_WORKBOOK` to its exact location. The notebook stops if it is missing.

In [6]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import pandas as pd, numpy as np, json, hashlib
ROOT=Path('/content/drive/MyDrive/03_Investigation_Explanation/Explanation_Evaluation')
INPUT_WORKBOOK=ROOT/'Input_Data'/'Agentic_GraphRAG_Full139_AI_Assessed_Final_QC.xlsx'
OUT=ROOT/'Results'/'Faithfulness_V1'
OUT.mkdir(parents=True,exist_ok=True)
assert INPUT_WORKBOOK.is_file(), f'Upload original QC workbook to: {INPUT_WORKBOOK}'
SEED=20261014
SAMPLE_N=40
print('Input:',INPUT_WORKBOOK)
print('Output:',OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Input: /content/drive/MyDrive/03_Investigation_Explanation/Explanation_Evaluation/Input_Data/Agentic_GraphRAG_Full139_AI_Assessed_Final_QC.xlsx
Output: /content/drive/MyDrive/03_Investigation_Explanation/Explanation_Evaluation/Results/Faithfulness_V1


## 2. Load claims and evidence; verify structure

In [7]:
claims=pd.read_excel(INPUT_WORKBOOK,sheet_name='Claim_Audit')
evidence=pd.read_excel(INPUT_WORKBOOK,sheet_name='Cited_Evidence')
required_claims=['Trigger ID','Partition','Instance','Risk types','Claim #','Claim text','Cited IDs']
required_evidence=['Trigger ID','Claim #','Evidence ID','Evidence text','Admissible']
assert set(required_claims).issubset(claims.columns),set(required_claims)-set(claims.columns)
assert set(required_evidence).issubset(evidence.columns),set(required_evidence)-set(evidence.columns)
KEY=['Trigger ID','Claim #']
assert not claims.duplicated(KEY).any(),'Duplicate claim identifiers'
claims['Claim #']=pd.to_numeric(claims['Claim #'],errors='raise').astype(int)
evidence['Claim #']=pd.to_numeric(evidence['Claim #'],errors='raise').astype(int)
assert len(claims)==155, f'Expected 155 claims, got {len(claims)}. Check workbook version.'
assert evidence['Evidence text'].notna().any(),'Evidence text missing'
print('Claims:',len(claims),'Cited evidence rows:',len(evidence))
print('Risk types:',claims['Risk types'].value_counts(dropna=False).to_dict())
print('Partitions:',claims['Partition'].value_counts(dropna=False).to_dict())

Claims: 155 Cited evidence rows: 572
Risk types: {'schedule': 48, 'scope': 41, 'dependency': 34, 'dependency, schedule': 13, 'schedule, scope': 7, 'dependency, scope': 6, 'dependency, schedule, scope': 6}
Partitions: {'validation': 85, 'exploratory_test': 70}


## 3. Reproducible sample of 40 claims
Stratify by risk type × partition, allocate proportionally with minimum-one where possible, then sample with a fixed seed. Previous support labels are **not** used for sampling, to avoid leaking AI judgments into independent review.

In [8]:
rng=np.random.default_rng(SEED)
d=claims.copy()
d['_stratum']=d['Risk types'].fillna('UNKNOWN').astype(str)+' | '+d['Partition'].fillna('UNKNOWN').astype(str)
counts=d['_stratum'].value_counts().sort_index()
assert SAMPLE_N>=len(counts), 'Sample too small for all strata'
alloc=pd.Series(1,index=counts.index,dtype=int)
remaining=SAMPLE_N-len(alloc)
while remaining:
    capacity=counts-alloc
    choices=capacity[capacity>0]
    assert len(choices),'Not enough unique claims'
    # Proportional balancing, deterministic tie break
    ratios=(counts.loc[choices.index]/alloc.loc[choices.index]).sort_values(ascending=False,kind='stable')
    alloc.loc[ratios.index[0]]+=1
    remaining-=1
parts=[]
for stratum,k in alloc.items():
    ix=d.index[d['_stratum'].eq(stratum)].to_numpy()
    parts.extend(rng.choice(ix,size=int(k),replace=False).tolist())
sampled=d.loc[parts].sort_values(['Trigger ID','Claim #']).copy()
assert len(sampled)==SAMPLE_N and not sampled.duplicated(KEY).any()
print('Sample size:',len(sampled))
print(sampled.groupby(['Risk types','Partition']).size().to_string())

Sample size: 40
Risk types                   Partition       
dependency                   exploratory_test    2
                             validation          6
dependency, schedule         exploratory_test    3
                             validation          1
dependency, schedule, scope  exploratory_test    1
                             validation          2
dependency, scope            exploratory_test    1
                             validation          1
schedule                     exploratory_test    4
                             validation          7
schedule, scope              exploratory_test    2
                             validation          1
scope                        exploratory_test    5
                             validation          4


## 4. Export blinded claim-review file and cited evidence
The review dataset contains claim text and *all associated evidence*. Invalid/missing citations remain visible as audit information; they are not silently treated as support. The review file excludes previous AI judgments.

In [9]:
ev=evidence.copy()
for c in ['Evidence ID','Evidence text','Admissible','Lookup source','Quality','Available at','Checkpoint']:
    if c not in ev: ev[c]=''
ev=ev.fillna('')
def make_evidence(g):
    lines=[]
    for _,r in g.iterrows():
        lines.append(f"[{r['Evidence ID']}] admissible={r['Admissible']} | source={r['Lookup source']} | quality={r['Quality']} | available={r['Available at']} | checkpoint={r['Checkpoint']}\n{r['Evidence text']}")
    return '\n\n'.join(lines)
byev=ev.groupby(KEY,dropna=False).apply(make_evidence,include_groups=False).rename('cited_evidence_text').reset_index()
review=sampled[required_claims].merge(byev,on=KEY,how='left',validate='one_to_one')
review['cited_evidence_text']=review['cited_evidence_text'].fillna('[NO MATCHING CITED EVIDENCE IN QC WORKBOOK]')
review=review.rename(columns={'Trigger ID':'trigger_id','Partition':'partition','Instance':'instance_id','Risk types':'risk_type','Claim #':'claim_number','Claim text':'claim_text','Cited IDs':'cited_ids'})
review['support_label']=''
review['reviewer_notes']=''
review['reviewer_id']=''
review['reviewed_at_utc']=''
review['sample_seed']=SEED
review_path=OUT/'claim_review_sample_40.csv'
# Avoid overwriting completed review on reruns
if review_path.exists():
    existing=pd.read_csv(review_path,keep_default_na=False)
    if 'support_label' in existing and existing['support_label'].astype(str).str.strip().ne('').any():
        raise RuntimeError('Existing review has labels. Move/archive it before regenerating sample.')
review.to_csv(review_path,index=False)
sampled[KEY+['_stratum']].to_csv(OUT/'sample_manifest_40.csv',index=False)
print('Saved blinded review:',review_path)
print('Rows with no matched cited evidence:',int(review['cited_evidence_text'].str.startswith('[NO MATCHING').sum()))

Saved blinded review: /content/drive/MyDrive/03_Investigation_Explanation/Explanation_Evaluation/Results/Faithfulness_V1/claim_review_sample_40.csv
Rows with no matched cited evidence: 0


## 5. Claim-review protocol
Review dataset: `claim_review_sample_40.csv`. Each review row contains **claim_text** and **cited_evidence_text**, with annotation fields: - `SUPPORTED`: All material parts of the claim are directly justified by admissible cited evidence.
- `PARTIALLY_SUPPORTED`: Some material parts supported; at least one substantive detail is not established.
- `UNSUPPORTED`: Evidence fails to establish the claim, contradicts it, or cited evidence is missing/invalid.
- `UNCERTAIN`: Cannot confidently decide from available evidence.

Also fill `reviewer_notes` and `reviewer_id` (e.g. `HUMAN_REVIEWER_1` or `AI_ASSISTED`). Do **not** call AI-generated labels independent human ground truth. Keep original file name and replace the exported CSV with your reviewed copy. The next cell only calculates final scores when all 40 labels are present.

## 6. Load completed review and calculate metrics (run after labeling)

In [10]:
reviewed=pd.read_csv(review_path,keep_default_na=False)
assert len(reviewed)==SAMPLE_N and not reviewed.duplicated(['trigger_id','claim_number']).any()
allowed={'SUPPORTED','PARTIALLY_SUPPORTED','UNSUPPORTED','UNCERTAIN'}
reviewed['support_label']=reviewed['support_label'].astype(str).str.strip().str.upper()
invalid=reviewed.loc[~reviewed['support_label'].isin(allowed),['trigger_id','claim_number','support_label']]
if len(invalid):
    print(f'INCOMPLETE: {len(invalid)} of {SAMPLE_N} claims have missing/invalid review labels.')
    display(invalid.head(12))
    print('No faithfulness score calculated. Complete the CSV and rerun this cell.')
else:
    assert reviewed['reviewer_id'].astype(str).str.strip().ne('').all(),'Fill reviewer_id for every row'
    counts=reviewed['support_label'].value_counts().reindex(sorted(allowed),fill_value=0)
    metrics={'sample_size':SAMPLE_N,'sample_seed':SEED,'label_counts':counts.to_dict(),
             'fully_supported_rate':float((reviewed['support_label']=='SUPPORTED').mean()),
             'partially_supported_rate':float((reviewed['support_label']=='PARTIALLY_SUPPORTED').mean()),
             'unsupported_rate':float((reviewed['support_label']=='UNSUPPORTED').mean()),
             'uncertain_rate':float((reviewed['support_label']=='UNCERTAIN').mean()),
             'reviewer_ids':reviewed['reviewer_id'].value_counts().to_dict(),
             'source_ground_truth_accuracy':'NOT_ESTABLISHED',
             'scope':'Faithfulness to cited graph-derived evidence only',
             'limitations':['40 sampled claims; estimates have sampling uncertainty',
                            'AI-assisted labels are not independent human validation',
                            'Evidence itself is not independently original-source verified',
                            'Stratified sample may not mirror overall claim prevalence']}
    pd.DataFrame([{'metric':k,'value':v} for k,v in metrics.items() if isinstance(v,(float,int))]).to_csv(OUT/'faithfulness_metrics.csv',index=False)
    (OUT/'faithfulness_summary.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')
    reviewed.to_csv(OUT/'claim_review_40_completed_snapshot.csv',index=False)
    print(json.dumps(metrics,indent=2))
    print('Saved final evaluation to:',OUT)

INCOMPLETE: 40 of 40 claims have missing/invalid review labels.


,trigger_id,claim_number,support_label
0,06850bb68bfc88e8562d,1,
1,157e5547098d1ea35ccd,1,
2,1a155d4b0711586aae70,1,
3,244077a603ece23e5b3a,2,
4,24c44eadb232cc328147,1,
5,254cff997b25295ced2b,1,
6,2bbed2d94dc34bc37dd0,1,
7,3959a8db7aa4406554cc,1,
8,4174c27e5bdc45718e45,1,
9,4ed5ef0afc48440f6a4b,1,


No faithfulness score calculated. Complete the CSV and rerun this cell.


## 7. Reporting and interpretation
Report **the numerator, denominator, reviewer provenance and limitations**. Example wording after labels exist: “In a reproducible 40-claim stratified review, X/40 claims were fully supported by their cited retrieved evidence; Y/40 were unsupported. Labels were [independently human reviewed / AI assisted]. This measures evidence faithfulness, not external factual accuracy.”

**Do not** describe 69.0% from the earlier full-155 AI assessment as independently human verified. If the sample remains unreviewed, report “faithfulness on the new sample not established.”